# Preparar una canción en un notebook GPU

Este notebook ejecuta el pipeline compartido dentro del runtime abierto, sin crear otra sesión Colab. Produce audio separado, letras editables y un ZIP del proyecto; **no renderiza el video final**.

**Estado:** notebook y paquete validados localmente. La inferencia completa en Google Colab y Kaggle sigue **sin verificar**. Se necesita Python 3.12+, GPU NVIDIA, FFmpeg y acceso a Internet para paquetes/modelos. En Kaggle activa GPU e Internet; en Colab selecciona un runtime GPU. Si tu imagen usa Python anterior, usa un runtime compatible, no reemplaces el Python del sistema.

Usa audio propio o autorizado. El modelo CTC tiene licencia CC-BY-NC-4.0. La GPU consume la cuota de tu cuenta. Descargarás los resultados antes de cerrar el runtime.


In [ ]:
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 12):  # noqa: UP036 - check the host before installing this package
    raise RuntimeError("Se necesita Python 3.12+. Selecciona una imagen compatible.")

# Sube el wheel construido desde una revisión revisada; no necesita acceso a GitHub.
WHEEL_PATH = Path("/content/lyrics_generator-0.1.0-py3-none-any.whl")
# Kaggle: /kaggle/input/TU_ENTRADA/lyrics_generator-0.1.0-py3-none-any.whl
OUTPUT_DIR = Path("/kaggle/working/karaoke") if Path("/kaggle/working").is_dir() else Path.cwd() / "karaoke-output"
print("Python:", sys.version.split()[0], "| Resultados:", OUTPUT_DIR)


## 1. Instalar desde un wheel local

El repositorio es privado. En una máquina donde ya tengas acceso, selecciona una revisión revisada y ejecuta `uv build`. Sube `dist/lyrics_generator-0.1.0-py3-none-any.whl` mediante Archivos en Colab o una entrada **privada** en Kaggle. Edita `WHEEL_PATH` arriba. No uses URLs con tokens ni pegues credenciales aquí.

La celda instala el wheel y sus dependencias del pipeline en una sola resolución; después `pip check` comprueba la consistencia del entorno. Descarga bibliotecas CUDA y puede tardar varios minutos. Usa un runtime nuevo para evitar conflictos con instalaciones anteriores. Si pide reiniciar el kernel, hazlo y repite la configuración y el diagnóstico antes de continuar.

Se fija Torch 2.10 con TorchCodec 0.10 y CUDA 12. El runtime debe soportar CUDA 12.8; no asumas compatibilidad con GPUs antiguas como P100. Si el entorno ya trae cuDNN 13, la celda se detiene sin desinstalar paquetes del proveedor. Reserva varios GB para dependencias, temporales y, después, modelos.


In [ ]:
import tempfile
import zipfile
from importlib.metadata import PackageNotFoundError, version

wheel_path = WHEEL_PATH.expanduser().resolve()
if wheel_path.suffix != ".whl" or not wheel_path.is_file():
    raise FileNotFoundError("Sube el wheel privado y configura WHEEL_PATH antes de instalar.")

# cuDNN 12 y 13 escriben las mismas bibliotecas; pip check no detecta el conflicto.
try:
    version("nvidia-cudnn-cu13")
except PackageNotFoundError:
    pass
else:
    raise RuntimeError("Este entorno ya tiene cuDNN 13. Elige un runtime limpio compatible con CUDA 12.")

# Lee solo el manifiesto incluido; no extrae ni ejecuta archivos del wheel.
with zipfile.ZipFile(wheel_path) as wheel:
    runtime_requirements = wheel.read("karaoke_app/runtime/requirements.txt")
with tempfile.TemporaryDirectory(prefix="karaoke-install-") as directory:
    requirements_path = Path(directory) / "requirements.txt"
    requirements_path.write_bytes(runtime_requirements)
    subprocess.run([
        sys.executable, "-m", "pip", "install", str(wheel_path), "-r", str(requirements_path),
    ], check=True)


In [ ]:
# Un fallo detiene el flujo: revisa el conflicto o empieza con un runtime limpio.
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)


In [ ]:
import json
from importlib import import_module

from karaoke_app.notebook import runtime_report

report = runtime_report()
print(json.dumps(report, indent=2))
if not all(report["commands"].get(name) for name in ("ffmpeg", "ffprobe", "nvidia-smi")):
    raise RuntimeError("Selecciona un runtime GPU con FFmpeg y ffprobe disponibles.")
if not all(report["modules"].values()):
    raise RuntimeError("Faltan dependencias del pipeline. Revisa la celda de instalación.")
subprocess.run(["nvidia-smi"], check=True)

torch = import_module("torch")

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch no puede usar la GPU. Revisa el runtime y sus bibliotecas CUDA.")


## 2. Elegir el audio

En Colab sube el audio usando el panel Archivos. En Kaggle añade tu archivo como entrada privada y copia su ruta bajo `/kaggle/input/`. Se aceptan archivos locales de audio/video, no URLs: las descargas de YouTube en estos runtimes pueden ser bloqueadas. Para YouTube usa la CLI local.

Edita `SOURCE`, título, artista e idioma. Si tienes una letra revisada, indica un TXT UTF-8 con un verso por línea. No pongas tokens ni cookies en estas celdas. `USE_LRCLIB = True` permite consultar metadatos y letras en servicios externos; desactívalo para transcribir sin búsqueda de referencias (los modelos siguen necesitando descarga).

Prefiere FLAC u otro archivo de audio: este frontend todavía no aplica la conversión audio-only de la CLI. Si eliges un video, el proyecto y el ZIP pueden conservar el video completo como fuente original.


In [ ]:
SOURCE = Path("/content/cancion.flac")  # Kaggle: /kaggle/input/TU_ENTRADA/cancion.flac
TITLE = "Mi canción"
ARTIST = "Artista"
LANGUAGE = "es"
LYRICS_TEXT = None  # O Path("/content/letra.txt")
USE_LRCLIB = True
WHISPER_MODEL = "large-v3"
DEMUCS_MODEL = "htdemucs_ft"


## 3. Preparar y revisar

Ejecuta esta celda cuando hayas configurado el audio. Cada ejecución tiene su propia carpeta: no sobrescribe proyectos anteriores. La alineación automática necesita revisión; una celda terminada no certifica precisión acústica. Si falla, el error indica dónde quedaron los diagnósticos y checkpoints.


In [ ]:
from karaoke_app.notebook import prepare_audio

result = prepare_audio(
    SOURCE, output_dir=OUTPUT_DIR, title=TITLE, artist=ARTIST, language=LANGUAGE,
    lyrics_text=LYRICS_TEXT, use_lrclib=USE_LRCLIB,
    whisper_model=WHISPER_MODEL, demucs_model=DEMUCS_MODEL,
)
print("Proyecto:", result.directory)
print("ZIP para descargar:", result.archive)


## 4. Descargar el proyecto

En Colab usa la celda siguiente. En Kaggle abre Output/Files y descarga el ZIP de la ruta mostrada. Conserva el proyecto completo. Revisa `vocals.flac` y `lyrics.lyricsfile.yaml` en LyricsEditor, reemplaza la letra corregida y usa la CLI para configurar y renderizar.

Al terminar, cierra el runtime desde la interfaz de Colab/Kaggle. Este notebook no administra ni detiene sesiones por ti.


In [ ]:
import importlib.util

if importlib.util.find_spec("google") and importlib.util.find_spec("google.colab"):
    from google.colab import files
    files.download(str(result.archive))
else:
    print("Descarga desde el panel de archivos:", result.archive)
